In [12]:
from getpass import getpass
from urllib.parse import quote_plus
from pymongo import MongoClient

USERNAME = "ac5231_db_user"                 
CLUSTER  = "cluster0.j0ohrxa.mongodb.net"   
DB_NAME  = "incidents"                      
password = "EHSK6plOSFyschwN"
uri = f"mongodb+srv://{USERNAME}:{password}@{CLUSTER}/?retryWrites=true&w=majority"


client = MongoClient(uri, serverSelectionTimeoutMS=5000)
client.admin.command("ping")               
db = client[DB_NAME]
print(f"\u2713 Connected to Atlas as {USERNAME!r} \u2014 using database {DB_NAME!r}")

✓ Connected to Atlas as 'ac5231_db_user' — using database 'incidents'


In [17]:
import pandas as pd
df = pd.DataFrame(list(db["incidents"].find()))
df.to_csv('coding_attempt_1.csv')

In [16]:
df['groups_by_coder'][0]['coder1']

[{'id': '1',
  'members': [{'role': 'actor', 'value': 'Individual journalist'},
   {'role': 'harm', 'value': 'Undermining of journalistic standards'},
   {'role': 'harm', 'value': 'Displacement of human judgement'},
   {'role': 'factor', 'value': 'Lack of transparency/disclosure'},
   {'role': 'factor',
    'value': 'Lack of internal transparency about AI-related management decisions'},
   {'role': 'factor', 'value': 'Lack of internal standards regarding AI use'},
   {'role': 'harmed_party', 'value': 'Journalist(s)'}]},
 {'id': '2',
  'members': [{'role': 'actor', 'value': 'News organization/publisher'},
   {'role': 'harm', 'value': 'Publication of low quality outputs / ai slop'}]}]

In [7]:
df.groups_by_coder[0]

{'coder1': [{'id': '1',
   'members': [{'role': 'actor', 'value': 'Individual journalist'},
    {'role': 'harm', 'value': 'Undermining of journalistic standards'},
    {'role': 'harm', 'value': 'Displacement of human judgement'},
    {'role': 'factor', 'value': 'Lack of transparency/disclosure'},
    {'role': 'factor',
     'value': 'Lack of internal transparency about AI-related management decisions'},
    {'role': 'factor', 'value': 'Lack of internal standards regarding AI use'},
    {'role': 'harmed_party', 'value': 'Journalist(s)'}]},
  {'id': '2',
   'members': [{'role': 'actor', 'value': 'News organization/publisher'},
    {'role': 'harm',
     'value': 'Publication of low quality outputs / ai slop'}]}],
 'coder2': [{'id': '1', 'members': [{'role': 'system', 'value': 'Editing'}]}]}

## By coder

Coding is stored per coder at `by_document.<doc_key>.by_coder.<coder>`, and each
coder's claim groups at `groups_by_coder.<coder>`. Documents coded before
multi-coder support sit flat on `by_document.<doc_key>` with a single `groups`
array, and are read back as the **first** coder's work.

The cell below flattens all of it into one tidy row per
**incident × document × coder × role × value** — the shape an intercoder
agreement statistic wants.


In [11]:
LEGACY_CODER = "coder1"   # must match the FIRST name in the app's CODERS env var
ROLES = ["actor", "harm", "factor", "harmed_party"]


def by_coder(entry):
    """{coder: coding} for one by_document entry, treating legacy flat coding
    (written before multi-coder support) as the first coder's."""
    nested = entry.get("by_coder")
    if isinstance(nested, dict):
        return nested
    if {"fields", "quotes", "roles"} & set(entry):
        return {LEGACY_CODER: entry}
    return {}


rows = []
for inc in db["incidents"].find():
    titles = {d.get("doc_id"): d.get("title") for d in (inc.get("documents") or [])}
    for doc_key, entry in (inc.get("by_document") or {}).items():
        for coder, coding in by_coder(entry).items():
            for role in ROLES:
                for value in (coding.get("roles") or {}).get(role, []):
                    rows.append({"incident_id": inc.get("incident_id"),
                                 "incident_title": inc.get("incident_title"),
                                 "doc_key": doc_key, "doc_title": titles.get(doc_key),
                                 "coder": coder, "role": role, "value": value})

tidy = pd.DataFrame(rows)
print(f"{len(tidy)} coded values from {tidy.coder.nunique() if len(tidy) else 0} coder(s)")
tidy.head(20)


200 coded values from 2 coder(s)


,incident_id,incident_title,doc_key,doc_title,coder,role,value
0,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,actor,News organization/publisher
1,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,actor,Individual journalist
2,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,harm,Publication of low quality outputs / ai slop
3,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,harm,Undermining of journalistic standards
4,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,harm,Displacement of human judgement
5,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,harm,Economic harm to publisher(s)/journalist(s)
6,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,harm,Increased workload / burden on journalists
7,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,factor,Lack of transparency/disclosure
8,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,factor,Lack of internal transparency about AI-related...
9,INC-001,Suncoast Searchlight editor AI use controversy,A5A5XZGY,Florida nonprofit news reporters ask board to ...,coder1,factor,Lack of internal standards regarding AI use
